In [1]:
import pandas as pd
import numpy as np
from ast import literal_eval
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. CARGA DE DATOS (El cimiento)
# Primero cargamos el archivo principal que te faltaba en la celda
df_movies = pd.read_csv('data/movies_metadata.csv', low_memory=False)
df_credits = pd.read_csv('data/credits.csv')
df_keywords = pd.read_csv('data/keywords.csv')

# 2. LIMPIEZA PREVENTIVA
# Eliminamos filas con IDs corruptos y convertimos a int para poder unirlos
df_movies = df_movies.drop([19730, 29503, 35587])
df_movies['id'] = df_movies['id'].astype(int)
df_credits['id'] = df_credits['id'].astype(int)
df_keywords['id'] = df_keywords['id'].astype(int)

# 3. FUSIÓN (Merge) - Aquí unimos las 3 tablas en una sola
df = df_movies.merge(df_credits, on='id')
df = df.merge(df_keywords, on='id')

# Solo trabajaremos con una muestra para no saturar la RAM de tu equipo
df = df.head(20000).copy()

# 4. PARSEO DE METADATOS
features = ['cast', 'crew', 'keywords', 'genres']
for feature in features:
    df[feature] = df[feature].apply(literal_eval)

def get_director(x):
    for i in x:
        if i['job'] == 'Director':
            return i['name']
    return np.nan

def get_list(x):
    if isinstance(x, list):
        names = [i['name'] for i in x]
        return names[:3] if len(names) > 3 else names
    return []

df['director'] = df['crew'].apply(get_director)
for feature in ['cast', 'keywords', 'genres']:
    df[feature] = df[feature].apply(get_list)

# 5. LIMPIEZA DE ESPACIOS (Tokenización)
def clean_data(x):
    if isinstance(x, list):
        return [str.lower(i.replace(" ", "")) for i in x]
    else:
        return str.lower(x.replace(" ", "")) if isinstance(x, str) else ''

for feature in ['cast', 'keywords', 'director', 'genres']:
    df[feature] = df[feature].apply(clean_data)

# 6. CREACIÓN DE LA "SOPA" Y VECTORIZACIÓN
def create_soup(x):
    return ' '.join(x['keywords']) + ' ' + ' '.join(x['cast']) + ' ' + x['director'] + ' ' + ' '.join(x['genres'])

df['soup'] = df.apply(create_soup, axis=1)

count = CountVectorizer(stop_words='english')
count_matrix = count.fit_transform(df['soup'])
cosine_sim2 = cosine_similarity(count_matrix, count_matrix)

# 7. FUNCIÓN DE RECOMENDACIÓN
df = df.reset_index()
indices = pd.Series(df.index, index=df['title'])

def get_recommendations(title, cosine_sim=cosine_sim2):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:11]
    movie_indices = [i[0] for i in sim_scores]
    return df['title'].iloc[movie_indices]

# PRUEBA FINAL
print("Recomendaciones para The Dark Knight Rises:")
print(get_recommendations('The Dark Knight Rises'))

Recomendaciones para The Dark Knight Rises:
12541      The Dark Knight
10170        Batman Begins
9271                Shiner
9834       Amongst Friends
7732              Mitchell
516      Romeo Is Bleeding
11411         The Prestige
10809       Helter Skelter
18866            Last Exit
4500       An Innocent Man
Name: title, dtype: object
